# PerfectPetClub - Data Engineering ETL Pipeline
## Customer Churn Prediction & Analytics

### 1. Project Overview & Business Problem
**PerfectPetClub** is an online subscription service providing pet food and supplies. A key business challenge is **customer churn**—when customers disable the auto-renewal feature of their subscription (`auto_renew = 0`).

This ETL pipeline extracts data across relational and transactional sources, validates data quality, engineers predictive features for churn analysis, and exports the final curated dataset into an optimized Apache Parquet format.

### 2. Architecture: Extract - Transform - Load (ETL)
* **Extract:** Ingest customer profiles, subscriptions, sales, payments, and product reviews from SQLite (`PerfectPetClub.db`).
* **Transform:**
  1. *Data Quality Validation:* Run automated assertion checks (null handling, date order logic, identifier uniqueness, boundary checks).
  2. *Feature Engineering:* Construct the binary churn target (`is_churned`), subscription tenure, financial aggregations, and customer feedback metrics.
  3. *Data Ethics & GDPR:* Mask/drop personally identifiable information (PII) such as passwords and street addresses.
* **Load:** Write the clean customer-level feature dataset to `output/perfectpetclub_churn_features.parquet`.

In [1]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import logging

# Ensure src modules are discoverable
sys.path.append('../src')

from extract_customer_subscriptions import (
    extract_customer_subscriptions,
    validate_customer_data,
    transform_customer_features
)
from extract_sales_reviews import (
    extract_sales_and_reviews,
    validate_sales_review_data
)

print('Environment and modules loaded successfully!')

Environment and modules loaded successfully!


## Stage 1: Data Extraction (SQLite Database)
We extract relational tables from `data/PerfectPetClub.db`:
* **Customer & Subscription profiles:** Customer identifiers, subscription tier, start/end dates, renewal status.
* **Transactions & Experience:** Sales orders, payment totals, and product reviews.

In [2]:
# Set database path relative to notebook directory
DB_PATH = '../data/PerfectPetClub.db'

# 1. Extract customer subscription records (Member A)
raw_customers = extract_customer_subscriptions(db_path=DB_PATH)
print(f'Extracted {len(raw_customers)} customer records.')
raw_customers.head(3)

2026-11-04 23:46:24,757 - INFO - Connecting to database: ../data/PerfectPetClub.db
2026-11-04 23:46:24,775 - INFO - Successfully extracted 221 customer records with subscription details.


Extracted 221 customer records.


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24


In [3]:
# 2. Extract aggregated sales, payments, and review metrics (Member B)
raw_sales_reviews = extract_sales_and_reviews(db_path=DB_PATH)
print(f'Extracted {len(raw_sales_reviews)} sales and review summaries.')
raw_sales_reviews.head(3)

2026-11-04 23:46:28,368 - INFO - Connecting to database for sales and reviews: ../data/PerfectPetClub.db
2026-11-04 23:46:28,414 - INFO - Successfully extracted financial and review metrics for 221 customers.


Extracted 221 sales and review summaries.


,customer_id,total_orders,total_spend_gbp,avg_order_value_gbp,most_recent_order_date,primary_payment_gateway,total_reviews_written,avg_star_rating,one_star_reviews_count,five_star_reviews_count
0,1,20,744.55,37.23,2026-01-10 20:43:36.234,paypool,10,2.40,4,1
1,2,253,10369.61,40.99,2026-11-09 21:35:20.498,cardpay,113,3.18,19,27
2,3,304,12846.92,42.26,2026-11-07 21:31:22.450,directdebit,134,3.08,28,31


,customer_id,total_orders,total_spend_gbp,avg_order_value_gbp,most_recent_order_date,primary_payment_gateway,total_reviews_written,avg_star_rating,one_star_reviews_count,five_star_reviews_count
0,1,20,744.55,37.23,2026-01-10 20:43:36.234,paypool,10,2.40,4,1
1,2,253,10369.61,40.99,2026-11-09 21:35:20.498,cardpay,113,3.18,19,27
2,3,304,12846.92,42.26,2026-11-07 21:31:22.450,directdebit,134,3.08,28,31


## Stage 2: Transform — Automated Data Quality Checks
Before performing feature engineering, we validate data consistency, integrity, and domain rules:
1. **Uniqueness:** Verify all `customer_id` values are unique.
2. **Missing Values:** Ensure primary identifiers and renewal statuses have zero nulls.
3. **Date Logic:** Check that `sub_start_date` <= `sub_end_date`.
4. **Value Boundaries:** Check non-negative spend values and review ratings bounded within [0, 5].

In [4]:
# Run automated data quality assertions
print('--- Validating Customer Data ---')
cust_dq = validate_customer_data(raw_customers)
for k, v in cust_dq.items():
    print(f'  ✓ {k}: {v}')

print('\n--- Validating Sales and Review Data ---')
sales_dq = validate_sales_review_data(raw_sales_reviews)
for k, v in sales_dq.items():
    print(f'  ✓ {k}: {v}')

2026-11-04 23:46:33,861 - INFO - Executing automated Data Quality checks on customer subscriptions...
2026-11-04 23:46:33,880 - INFO - All Customer Data Quality checks PASSED successfully (4/4).
2026-11-04 23:46:33,881 - INFO - Executing automated Data Quality checks on sales and reviews...
2026-11-04 23:46:33,882 - INFO - All Sales/Review Data Quality checks PASSED successfully (3/3).


--- Validating Customer Data ---
  ✓ null_checks: {'customer_id': 0, 'email': 0, 'subscription_id': 0, 'auto_renew': 0}
  ✓ is_unique: True
  ✓ valid_auto_renew: True
  ✓ date_logic_valid: True

--- Validating Sales and Review Data ---
  ✓ valid_spend: True
  ✓ valid_orders: True
  ✓ valid_ratings: True


2026-11-04 23:25:05,335 - INFO - Executing automated Data Quality checks on sales and reviews...


2026-11-04 23:25:05,335 - INFO - All Sales/Review Data Quality checks PASSED successfully (3/3).


--- Validating Customer Data ---
  ✓ null_checks: {'customer_id': 0, 'email': 0, 'subscription_id': 0, 'auto_renew': 0}
  ✓ is_unique: True
  ✓ valid_auto_renew: True
  ✓ date_logic_valid: True

--- Validating Sales and Review Data ---
  ✓ valid_spend: True
  ✓ valid_orders: True
  ✓ valid_ratings: True


## Stage 3: Transform — Feature Engineering & Integration
In this stage, we construct the predictive features for churn analysis:
* `is_churned`: **Target variable** (`1` if `auto_renew == 0` else `0`).
* `subscription_tenure_days`: Length of customer relationship in days.
* `avg_order_value_gbp`: Average transaction size.
* `one_star_reviews_count`: Dissatisfaction indicator.
* `postcode_district`: Area code for geographic segmentation.

In [5]:
# 1. Transform customer features
clean_customers = transform_customer_features(raw_customers)

# 2. Merge all features on customer_id (1-to-1 relationship)
master_features = clean_customers.merge(raw_sales_reviews, on='customer_id', how='left')

# 3. Clean up PII for GDPR compliance (drop name, full postcode, and raw subscription IDs)
final_dataset = master_features.drop(columns=['customer_name', 'postcode', 'subscription_id'])

print(f'Master feature dataset shape: {final_dataset.shape}')
final_dataset.head()

2026-11-04 23:46:42,020 - INFO - Applying feature engineering on customer subscriptions...
2026-11-04 23:46:42,034 - INFO - Target Engineered: 86/221 customers churned (38.91%).


Master feature dataset shape: (221, 20)


,customer_id,email,county,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_district,total_orders,total_spend_gbp,avg_order_value_gbp,most_recent_order_date,primary_payment_gateway,total_reviews_written,avg_star_rating,one_star_reviews_count,five_star_reviews_count
0,1,Sargsyan701@somemail.co.uk,Berkshire,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72,1,30,SL6,20,744.55,37.23,2026-01-10 20:43:36.234,paypool,10,2.40,4,1
1,2,Mammadova282@example.com,Oxfordshire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72,0,330,OX7,253,10369.61,40.99,2026-11-09 21:35:20.498,cardpay,113,3.18,19,27
2,3,Sana858@notrealemail.com,Worcestershire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,DY10,304,12846.92,42.26,2026-11-07 21:31:22.450,directdebit,134,3.08,28,31
3,4,Omar.Farooq4@example.com,Kent,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,TN9,1055,95775.04,90.78,2026-11-09 21:56:40.499,paypool,537,3.30,83,144
4,5,Alejandro357@somemail.co.uk,Hertfordshire,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72,1,120,WD7,44,1797.38,40.85,2026-04-12 19:52:46.436,cardpay,18,3.56,1,5


,customer_id,email,county,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_district,total_orders,total_spend_gbp,avg_order_value_gbp,most_recent_order_date,primary_payment_gateway,total_reviews_written,avg_star_rating,one_star_reviews_count,five_star_reviews_count
0,1,Sargsyan701@somemail.co.uk,Berkshire,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72,1,30,SL6,20,744.55,37.23,2026-01-10 20:43:36.234,paypool,10,2.40,4,1
1,2,Mammadova282@example.com,Oxfordshire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72,0,330,OX7,253,10369.61,40.99,2026-11-09 21:35:20.498,cardpay,113,3.18,19,27
2,3,Sana858@notrealemail.com,Worcestershire,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,DY10,304,12846.92,42.26,2026-11-07 21:31:22.450,directdebit,134,3.08,28,31
3,4,Omar.Farooq4@example.com,Kent,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,TN9,1055,95775.04,90.78,2026-11-09 21:56:40.499,paypool,537,3.30,83,144
4,5,Alejandro357@somemail.co.uk,Hertfordshire,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72,1,120,WD7,44,1797.38,40.85,2026-04-12 19:52:46.436,cardpay,18,3.56,1,5


### Feature Engineering Summary & Churn Analysis
Quick validation of churn distribution and key drivers:

In [6]:
# Churn rate by subscription tier
tier_summary = final_dataset.groupby('subscription_tier')['is_churned'].agg(
    total_customers='count',
    churn_rate='mean'
).reset_index()

tier_summary['churn_rate'] = (tier_summary['churn_rate'] * 100).round(1).astype(str) + '%'
print('--- Churn Rate by Subscription Tier ---')
print(tier_summary)

# Churned vs Retained spend & review comparison
print('\n--- Mean Metrics by Churn Status ---')
print(final_dataset.groupby('is_churned')[['total_spend_gbp', 'total_orders', 'avg_star_rating', 'one_star_reviews_count']].mean().round(2))

--- Churn Rate by Subscription Tier ---
  subscription_tier  total_customers churn_rate
0             Basic               77      33.8%
1           Premium               47      40.4%
2        SuperSaver               97      42.3%

--- Mean Metrics by Churn Status ---
            total_spend_gbp  total_orders  avg_star_rating  \
is_churned                                                   
0                   5598.25        121.86             3.17   
1                   3521.85         78.34             3.20   

            one_star_reviews_count  
is_churned                          
0                             8.59  
1                             5.71  


## Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file.
Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.

In [7]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')

✓ Successfully loaded 221 records into Parquet format:
  Location: /Users/srijana/Downloads/data-engineering/DataEngineeringPresentation/output/perfectpetclub_churn_features.parquet
  File size: 33,697 bytes


In [8]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

Parquet verification passed: 221 rows, 20 columns.

Dataset Columns:
 - customer_id (int64)
 - email (object)
 - county (object)
 - sub_start_date (object)
 - sub_end_date (object)
 - auto_renew (int64)
 - subscription_tier (object)
 - delivery_service (object)
 - is_churned (int64)
 - subscription_tenure_days (int64)
 - postcode_district (object)
 - total_orders (int64)
 - total_spend_gbp (float64)
 - avg_order_value_gbp (float64)
 - most_recent_order_date (object)
 - primary_payment_gateway (object)
 - total_reviews_written (int64)
 - avg_star_rating (float64)
 - one_star_reviews_count (int64)
 - five_star_reviews_count (int64)
